In [ ]:
!pip install pymupdf

In [ ]:
!pip install requests langchain openai crewai python-dotenv

In [ ]:
!pip install pymupdf crewai crewai-tools

In [ ]:
# Paso 0: Cargar la API Key de forma segura y configurar entorno
from openai import OpenAI
from getpass import getpass
import os

api_key = getpass("🔑 Ingresa tu API key de OpenAI: ").strip()
client = OpenAI(api_key=api_key)
os.environ["OPENAI_API_KEY"] = api_key

🔑 Ingresa tu API key de OpenAI: ··········


In [ ]:
import os
from google.colab import files

In [ ]:
os.makedirs("pdfs", exist_ok=True)
uploaded = files.upload()  # Subí aquí tus 2 PDFs

Saving Documento IA.pdf to Documento IA.pdf
Saving Q_36_ (2023)_03.pdf to Q_36_ (2023)_03.pdf


In [ ]:
# Mover archivos subidos a la carpeta "pdfs"
import shutil
for filename in uploaded.keys():
    shutil.move(filename, f"pdfs/{filename}")


In [ ]:
# Paso 3: Extraer texto de cada PDF y guardarlo como .txt
import fitz  # PyMuPDF

os.makedirs("txts", exist_ok=True)

def extract_text_from_pdf(pdf_path, output_txt_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write(text)

In [ ]:
# Convertir todos los PDFs en "pdfs/"
for pdf_file in os.listdir("pdfs"):
    if pdf_file.lower().endswith(".pdf"):
        pdf_path = os.path.join("pdfs", pdf_file)
        txt_path = os.path.join("txts", pdf_file.replace(".pdf", ".txt"))
        extract_text_from_pdf(pdf_path, txt_path)


In [ ]:
# Paso 4: Usar DirectoryReadTool
from crewai_tools import DirectoryReadTool
from crewai import Agent, Task, Crew

docs_tool = DirectoryReadTool(directory="./txts")

In [ ]:
from crewai import Agent
from langchain.chat_models import ChatOpenAI

llm_gpt35 = ChatOpenAI(model="gpt-3.5-turbo")

agent = Agent(
    role='Analista de documentos',
    goal='Leer y resumir el contenido de varios documentos',
    tools=[docs_tool],
    backstory='Experto en lectura de informes PDF y generación de resúmenes.',
    llm=llm_gpt35
)


In [ ]:
from langchain.chat_models import ChatOpenAI
from crewai import Agent, Task, Crew
from crewai_tools import DirectoryReadTool

docs_tool = DirectoryReadTool(directory="./txts")
llm_gpt35 = ChatOpenAI(model="gpt-3.5-turbo")

agent = Agent(
    role='Analista de documentos',
    goal='Leer y resumir el contenido de varios documentos',
    tools=[docs_tool],
    backstory='Experto en lectura de informes PDF y generación de resúmenes.',
    llm=llm_gpt35
)

task = Task(
    description='Lee todos los documentos de texto en la carpeta y crea un resumen de cada uno.',
    expected_output='Resumen de cada documento, separado por títulos.',
    agent=agent
)

crew = Crew(tasks=[task])
resultado = crew.kickoff()
print(resultado)


- Resumen del documento "Q_36_ (2023)_03.txt":
El documento "Q_36_ (2023)_03.txt" trata sobre el impacto de la inteligencia artificial en la sociedad actual y futura. Aborda temas como la automatización de tareas, la ética en el uso de la IA y las posibles implicaciones en el mercado laboral.

- Resumen del documento "Documento IA.txt":
El documento "Documento IA.txt" presenta un análisis detallado sobre los algoritmos de inteligencia artificial utilizados en aplicaciones del mundo real. Se discuten ejemplos de implementaciones exitosas y se mencionan posibles desafíos éticos y de privacidad asociados con la IA.


In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import DirectoryReadTool
from langchain.chat_models import ChatOpenAI

# Reutilizamos la herramienta y el modelo
docs_tool = DirectoryReadTool(directory="./txts")
llm_gpt35 = ChatOpenAI(model="gpt-3.5-turbo")

# Crear el agente nuevamente
qa_agent = Agent(
    role='Especialista en comprensión de documentos',
    goal='Responder preguntas sobre el contenido de documentos',
    tools=[docs_tool],
    backstory='Tiene experiencia respondiendo preguntas complejas basadas en múltiples documentos.',
    llm=llm_gpt35
)

# ✅ Pregunta del usuario (puede ser dinámica)
pregunta_usuario = "¿Cuáles son los principales hallazgos del informe 1?"

qa_task = Task(
    description=f"Respondé a la siguiente pregunta usando el contenido de los archivos: {pregunta_usuario}",
    expected_output='Respuesta clara y fundamentada en los documentos.',
    agent=qa_agent
)

# Ejecutar el agente con la pregunta
qa_crew = Crew(tasks=[qa_task])
respuesta = qa_crew.kickoff()

print("📌 Respuesta del agente:\n")
print(respuesta)


📌 Respuesta del agente:

I tried reusing the same input, I must stop using this action input. I'll try something else instead.




In [ ]:
from crewai_tools import DirectoryReadTool

# Asegurarse que apunta a la carpeta correcta en tiempo de ejecución
docs_tool = DirectoryReadTool(directory="./txts")

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import DirectoryReadTool
from langchain.chat_models import ChatOpenAI
import os

# Verificar que hay archivos de texto
txt_dir = "./txts"
txt_files = [f for f in os.listdir(txt_dir) if f.endswith(".txt")]

if not txt_files:
    raise FileNotFoundError("❌ No se encontraron archivos .txt en la carpeta './txts'. Asegúrate de haber convertido los PDFs correctamente.")

print("✅ Archivos disponibles para análisis:")
for f in txt_files:
    print(f" - {f}")

# Mostrar primeros 300 caracteres de cada archivo para confirmar contenido
print("\n🔎 Previsualización de los documentos:\n")
for f in txt_files:
    with open(os.path.join(txt_dir, f), "r", encoding="utf-8") as file:
        content = file.read()
        preview = content[:300].replace("\\n", " ").replace("  ", " ")
        print(f"📄 {f} → {preview[:300]}...\n")

# Crear la herramienta y modelo
docs_tool = DirectoryReadTool(directory=txt_dir)
llm_gpt35 = ChatOpenAI(model="gpt-3.5-turbo")

# Crear el agente
qa_agent = Agent(
    role='Especialista en comprensión documental',
    goal='Responder preguntas complejas con base en múltiples documentos',
    tools=[docs_tool],
    backstory='Con experiencia en análisis de informes técnicos, académicos y legales, brinda respuestas fundamentadas, claras y concisas.',
    llm=llm_gpt35
)

# Pregunta dinámica del usuario
pregunta_usuario = input("❓ Ingresá tu pregunta sobre los documentos: ").strip()

qa_task = Task(
    description=f"""Usando exclusivamente el contenido de los documentos disponibles,
respondé a la siguiente consulta del usuario: "{pregunta_usuario}".
Proporcioná una respuesta clara, basada en evidencia textual, y si corresponde, mencioná a qué documento te referís.""",
    expected_output='Respuesta precisa y basada en los documentos analizados.',
    agent=qa_agent
)

# Ejecutar el agente
print("\n🧠 Procesando la información con el agente inteligente...\n")
qa_crew = Crew(tasks=[qa_task])
respuesta = qa_crew.kickoff()

# Mostrar respuesta
print("📌 Respuesta del agente:\n")
print(respuesta)


✅ Archivos disponibles para análisis:
 - Q_36_ (2023)_03.txt
 - Documento IA.txt

🔎 Previsualización de los documentos:

📄 Q_36_ (2023)_03.txt → REVISTA QURRICULUM, ABRIL 36; 2023, PP. 51-60
51
DOI: https://doi.org/10.25145/j.qurricul.2023.36.03
Revista Qurriculum, 36; julio 2023, pp. 51-60; ISSN: e-2530-8386
EL IMPACTO DE LA INTELIGENCIA ARTIFICIAL 
EN LA EDUCACIÓN: TRANSFORMACIÓN DE 
LA FORMA DE ENSEÑAR Y DE APRENDER
Carina S. González-Gon...

📄 Documento IA.txt → La inteligencia artificial
en la educación
Consejo Dirección Sectorial
Directivo de Planificación Educativa
Exp. 1-483/24, del 21 de marzo de 2024
Consejo Dirección Sectorial
Directivo de Planificación Educativa
Exp. 1-483/24, del 21 de marzo de 2024
La inteligencia artificial
en la educación
Consej...

❓ Ingresá tu pregunta sobre los documentos: ¿Cual es el impacto de la IA en la educación?

🧠 Procesando la información con el agente inteligente...

📌 Respuesta del agente:

I tried reusing the same input, I must stop using 

In [ ]:
# Crear un modelo con temperatura baja para respuestas más deterministas y precisas
llm_gpt35 = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)

# Redefinir el agente con mayor contexto, propósito y capacidad crítica
qa_agent = Agent(
    role="Consultor documental senior en IA",
    goal=(
        "Responder con precisión, profundidad y claridad a preguntas basadas en contenido documental. "
        "Debe identificar hallazgos clave, patrones importantes y citas relevantes cuando sea posible. "
        "Si el contenido es insuficiente o ambiguo, debe decirlo explícitamente."
    ),
    tools=[docs_tool],
    backstory=(
        "Este agente ha sido entrenado en análisis de documentos extensos, como reportes técnicos, "
        "informes académicos, investigaciones jurídicas y papers científicos. "
        "Tiene una sólida capacidad para resumir, interpretar y responder preguntas complejas "
        "utilizando sólo la información proporcionada en los archivos disponibles."
    ),
    llm=llm_gpt35,
    verbose=True,  # opcional: muestra trazas si querés depurar en clase
    allow_delegation=False  # asegura que el agente asuma toda la responsabilidad
)


In [ ]:
# Ejecutar el agente
print("\n🧠 Procesando la información con el agente inteligente...\n")
qa_crew = Crew(tasks=[qa_task])
respuesta = qa_crew.kickoff()

# Mostrar respuesta
print("📌 Respuesta del agente:\n")
print(respuesta)


🧠 Procesando la información con el agente inteligente...

📌 Respuesta del agente:

Thought:


In [ ]:
# 🔁 Loop interactivo para hacer preguntas al agente
def preguntar_sobre_documentos():
    print("🔎 Podés hacer preguntas sobre los documentos cargados.")
    print("✏️ Escribí tu pregunta y presioná Enter. Escribí 'salir' para finalizar.\n")

    while True:
        pregunta = input("❓ Pregunta: ").strip()
        if pregunta.lower() == "salir":
            print("👋 Finalizando sesión de preguntas.")
            break
        if not pregunta:
            print("⚠️ Pregunta vacía. Intentá de nuevo.")
            continue

        # Descripción mejorada, más precisa y orientada al análisis textual
        description = (
            f"Tu objetivo es responder a la siguiente consulta del usuario usando exclusivamente el contenido "
            f"de los documentos dentro de ./txts: \"{pregunta}\".\n\n"
            "Debés leer y analizar el contenido disponible. Si hay estadísticas, citas, comparaciones o ejemplos "
            "relevantes, incluílos. Si no hay suficiente información para responder, explicá esa limitación de forma clara."
        )

        task = Task(
            description=description,
            expected_output="Una respuesta detallada, precisa y bien justificada basada en los documentos.",
            agent=qa_agent
        )

        print("\n🤖 Procesando la respuesta...\n")
        crew = Crew(tasks=[task])
        respuesta = crew.kickoff()

        print("💬 Respuesta del agente:\n")
        print(respuesta)
        print("\n──────────────────────────────────────\n")



In [ ]:
preguntar_sobre_documentos()


🔎 Podés hacer preguntas sobre los documentos cargados.
✏️ Escribí tu pregunta y presioná Enter. Escribí 'salir' para finalizar.

❓ Pregunta: ¿Cual es impacto de la Ia en los colegios incial y primaria del Perú? 

🤖 Procesando la respuesta...

# Agent: Consultor documental senior en IA
## Task: Tu objetivo es responder a la siguiente consulta del usuario usando exclusivamente el contenido de los documentos dentro de ./txts: "¿Cual es impacto de la Ia en los colegios incial y primaria del Perú?".

Debés leer y analizar el contenido disponible. Si hay estadísticas, citas, comparaciones o ejemplos relevantes, incluílos. Si no hay suficiente información para responder, explicá esa limitación de forma clara.


# Agent: Consultor documental senior en IA
## Thought: Dado que la pregunta se centra en el impacto de la IA en los colegios de educación inicial y primaria en Perú, buscaré información relevante en los documentos disponibles para identificar cualquier referencia específica a este tema

KeyboardInterrupt: Interrupted by user